# Oracle Gap Scan (Employed Only): per-group (baseline - oracle) for all feature subsets

Filtered to ESR==1, WKW==1 (year-round civilian employed).
WKHP excluded from features (used in filtering, mechanical relationship with log-income).

Features: AGEP, SCHL, MARC, RELPC (4 features, 15 subsets).

Runs both `sex` and `race_binary` group_by settings.

In [1]:
import numpy as np
import pandas as pd
import importlib
import warnings
from itertools import combinations
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

import folktables_utils
import train_methods
importlib.reload(folktables_utils)
importlib.reload(train_methods)

from folktables_utils import load_acs_income
from train_methods import train_group_oracles, compute_baseline_losses

In [2]:
# ====== CONFIGURATION ======
YEAR = 2018
ALL_FEATURES = ["AGEP", "SCHL", "MARC", "RELPC"]

ALL_STATES = [
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
]

GROUP_BY_LIST = ["sex", "race_binary"]
# ===========================

In [3]:
# Enumerate all non-empty subsets
all_subsets = []
for k in range(1, len(ALL_FEATURES) + 1):
    for combo in combinations(ALL_FEATURES, k):
        all_subsets.append(combo)

print(f"Features: {ALL_FEATURES}")
print(f"Total subsets: {len(all_subsets)}")

Features: ['AGEP', 'SCHL', 'MARC', 'RELPC']
Total subsets: 15


In [4]:
for GROUP_BY in GROUP_BY_LIST:
    print(f"\n{'='*60}")
    print(f"GROUP_BY = {GROUP_BY} (employed_only=True)")
    print(f"{'='*60}\n")

    feat_to_idx = {f: i for i, f in enumerate(ALL_FEATURES)}
    rows = []
    save_path = f"../oracle_gap_scan_{GROUP_BY}_employed.csv"

    for si, state in enumerate(ALL_STATES):
        print(f"[{si+1}/{len(ALL_STATES)}] {state} ", end="", flush=True)

        X_all, y, g, num_groups, _ = load_acs_income(
            states=(state,), year=YEAR, group_by=GROUP_BY,
            features=ALL_FEATURES, employed_only=True
        )
        scaler = StandardScaler()
        X_all_scaled = scaler.fit_transform(X_all)
        baseline = compute_baseline_losses(y, g, num_groups)

        for subset in all_subsets:
            col_idx = [feat_to_idx[f] for f in subset]
            X_sub = X_all_scaled[:, col_idx]

            oracle = train_group_oracles(
                X_sub, y, g, num_groups=num_groups, closed_form=True, verbose=False
            )

            gap_g0 = baseline[0] - oracle[0]
            gap_g1 = baseline[1] - oracle[1]
            gap_ratio = gap_g0 / gap_g1 if gap_g1 > 1e-10 else np.inf

            rows.append({
                'state': state,
                'features': ','.join(subset),
                'n_features': len(subset),
                'baseline_g0': baseline[0],
                'baseline_g1': baseline[1],
                'oracle_g0': oracle[0],
                'oracle_g1': oracle[1],
                'gap_g0': gap_g0,
                'gap_g1': gap_g1,
                'gap_ratio_g0_g1': gap_ratio,
            })

        df = pd.DataFrame(rows)
        df.to_csv(save_path, index=False)
        print(f"OK ({len(all_subsets)} subsets, {len(rows)} rows total)")

    print(f"\nDone. Saved to {save_path}")


GROUP_BY = sex (employed_only=True)

[1/50] AL Removed 25329 samples with NaN values
Filtered to full-time year-round employed (ESR==1, WKHP>=35, WKW==1): 22448 -> 14286
Removed 6 samples with non-positive income
Selected 4 features: ['AGEP', 'SCHL', 'MARC', 'RELPC']
Loaded 14280 samples, 4 features, 2 groups (group_by=sex)
OK (15 subsets, 15 rows total)
[2/50] AK Removed 3131 samples with NaN values
Filtered to full-time year-round employed (ESR==1, WKHP>=35, WKW==1): 3580 -> 1765
Selected 4 features: ['AGEP', 'SCHL', 'MARC', 'RELPC']
Loaded 1765 samples, 4 features, 2 groups (group_by=sex)
OK (15 subsets, 30 rows total)
[3/50] AZ Removed 36455 samples with NaN values
Filtered to full-time year-round employed (ESR==1, WKHP>=35, WKW==1): 33535 -> 20531
Removed 6 samples with non-positive income
Selected 4 features: ['AGEP', 'SCHL', 'MARC', 'RELPC']
Loaded 20525 samples, 4 features, 2 groups (group_by=sex)
OK (15 subsets, 45 rows total)
[4/50] AR Removed 16460 samples with NaN values
F

## Analysis

In [5]:
cols = ['state', 'features', 'n_features', 'gap_g0', 'gap_g1', 'gap_ratio_g0_g1']

for GROUP_BY in GROUP_BY_LIST:
    print(f"\n{'='*60}")
    print(f"GROUP_BY = {GROUP_BY} (employed_only)")
    print(f"{'='*60}")

    df = pd.read_csv(f"../oracle_gap_scan_{GROUP_BY}_employed.csv")
    print(f"Loaded {len(df)} rows")

    print(f"\n--- Largest gap_ratio (G0 benefits much more than G1) ---")
    print(df.sort_values('gap_ratio_g0_g1', ascending=False)[cols].head(20).to_string(index=False))

    print(f"\n--- Smallest gap_ratio (G1 benefits much more than G0) ---")
    print(df.sort_values('gap_ratio_g0_g1', ascending=True)[cols].head(20).to_string(index=False))


GROUP_BY = sex (employed_only)
Loaded 750 rows

--- Largest gap_ratio (G0 benefits much more than G1) ---
state        features  n_features   gap_g0   gap_g1  gap_ratio_g0_g1
   AZ      MARC,RELPC           2 0.052413 0.022585         2.320674
   VA            MARC           1 0.051236 0.023525         2.177964
   HI            MARC           1 0.031651 0.014618         2.165315
   VA      MARC,RELPC           2 0.067375 0.031205         2.159129
   NV            MARC           1 0.028755 0.013396         2.146546
   ME      MARC,RELPC           2 0.046996 0.022010         2.135256
   MD            MARC           1 0.046743 0.022119         2.113302
   AK            MARC           1 0.036939 0.017662         2.091404
   VT      AGEP,RELPC           2 0.033947 0.016258         2.088013
   NY            MARC           1 0.025346 0.012151         2.085851
   VT AGEP,MARC,RELPC           3 0.044040 0.021246         2.072823
   AZ            MARC           1 0.033227 0.016354         2.031